# cap10_state_conditioned_trend_participation_controller_v1

Purpose: test whether the latest `cap10_release_oos_poor_window_diagnostic_v1` failure can be improved by a small, state-conditioned participation lift with an explicit equity drawdown controller before opening another PPO branch.

Governance:
- Canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1` runtime composer family.
- Direct comparison baseline: `sparse_release_cap10_rebound_balance_v1` and its follow-up diagnostic `cap10_release_oos_poor_window_diagnostic_v1`.
- Role: `diagnose`; this is a path-level replay screen, not a runtime promotion.
- Changed layer: `model_selection` / `training-objective diagnosis`; no actor weights and no broker execution path change.
- Target windows: all 12 cap10 release daily paths from `wf_2022` and `wf_2023_2024`.
- Acceptance: each scored path must clear `relative_return >= +30pp` and `MaxDD <= 15%`; shared poor-window clusters must avoid low participation without reopening drawdown.
- Failure mode: a post-hoc floor improves participation but consumes the drawdown budget, which means the next step must be a training objective or checkpoint selector rather than a runtime overlay.

External references used for discipline:
- Qlib GitHub and paper for reproducible research artifact discipline: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- Probability of Backtest Overfitting / CSCV for limiting this to a small diagnostic grid: https://doi.org/10.21314/JCF.2016.322
- Deflated Sharpe Ratio for multiple-testing caution when selecting from parameter screens: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551
- Purged/embargoed time-series CV as the rule for future training labels derived from this screen: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2708678


In [1]:
from __future__ import annotations

import json
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "AGENTS.md").exists() and (path / "src/soxl_quant").exists():
            return path
    raise RuntimeError("could not locate repository root")


ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.budget_counterfactual import dashboard_daily_frame
from src.soxl_quant.app.oos_diagnostics import load_qlib_feature_frame
from src.soxl_quant.app.participation_counterfactual import (
    ReentryParticipationOverlayConfig,
    apply_reentry_participation_overlay,
)
from src.soxl_quant.app.state_conditioned_objective import (
    StateConditionedObjectiveConfig,
    daily_path_metrics,
    render_state_conditioned_objective_review,
    score_state_conditioned_objective,
    slice_daily_path_metrics,
    summarize_state_conditioned_objective,
)

EXPERIMENT = "cap10_state_conditioned_trend_participation_controller_v1"
CARD = ROOT / "runtime/model_selection/card" / EXPERIMENT
SOURCE_CARD = ROOT / "runtime/model_selection/card/cap10_release_oos_poor_window_diagnostic_v1"
SELECTED_DAILY_DIR = CARD / "selected_daily_paths"
CARD.mkdir(parents=True, exist_ok=True)
SELECTED_DAILY_DIR.mkdir(parents=True, exist_ok=True)

TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
CLUSTER_PARTICIPATION_FLOOR = 0.25

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


## 1. Load Current Evidence

This uses the already-executed cap10 release diagnostic as the source of truth. The screen does not re-run PPO and does not update `docs/active_workpad.md`.

In [2]:
manifest = pd.read_csv(SOURCE_CARD / "daily_path_manifest.csv")
source_metrics = pd.read_csv(SOURCE_CARD / "daily_path_metrics.csv")
clusters = pd.read_csv(SOURCE_CARD / "aggregate/poor_window_clusters.csv")
gate_gap = pd.read_csv(SOURCE_CARD / "aggregate/gate_gap_attribution.csv")
feature_consensus = pd.read_csv(SOURCE_CARD / "aggregate/feature_group_consensus.csv")

first_dates = []
last_dates = []
for csv_path in manifest["daily_csv"]:
    daily = pd.read_csv(ROOT / csv_path, parse_dates=["datetime"])
    first_dates.append(daily["datetime"].min())
    last_dates.append(daily["datetime"].max())
feature_start = min(first_dates).date().isoformat()
feature_end = max(last_dates).date().isoformat()
features = load_qlib_feature_frame(feature_start, feature_end)

required_feature_cols = [
    "mom_21",
    "relative_strength_21",
    "vol_of_vol_21",
    "tracking_error_vol_21",
    "vix_percentile_252",
]
missing = [col for col in required_feature_cols if col not in features.columns]
if missing:
    raise ValueError(f"feature frame missing required columns: {missing}")

context = {
    "source_card": str(SOURCE_CARD.relative_to(ROOT)),
    "daily_path_count": int(len(manifest)),
    "feature_start": feature_start,
    "feature_end": feature_end,
    "feature_rows": int(len(features)),
    "dominant_gate_failure": gate_gap["failure_type"].value_counts().idxmax(),
    "dominant_cluster_failure": clusters["dominant_failure_type"].value_counts().idxmax(),
}
print(json.dumps(context, indent=2))
display(source_metrics)
display(clusters)
display(feature_consensus.head(5))


[23643:MainThread](2026-05-26 06:16:32,633) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[23643:MainThread](2026-05-26 06:16:33,434) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[23643:MainThread](2026-05-26 06:16:33,435) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[23643:MainThread](2026-05-26 06:16:36,092) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Loading data Done


[23643:MainThread](2026-05-26 06:16:36,095) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[23643:MainThread](2026-05-26 06:16:36,097) INFO - qlib.timer - [log.py:127] - Time cost: 0.007s | Init data Done


{
  "source_card": "runtime/model_selection/card/cap10_release_oos_poor_window_diagnostic_v1",
  "daily_path_count": 12,
  "feature_start": "2022-01-03",
  "feature_end": "2024-12-30",
  "feature_rows": 752,
  "dominant_gate_failure": "underparticipation",
  "dominant_cluster_failure": "underparticipation"
}


,label,target_variant,release_cap,fold,return,buy_and_hold_return,relative_return,sharpe,max_dd,avg_pos,gate_pass
0,cap10_release_0.10__wf_2022,cap10_release_0.10,0.10,wf_2022,-10.137677,-76.614257,72.537447,-0.767384,11.809271,0.171053,True
1,cap10_release_0.10__wf_2023_2024,cap10_release_0.10,0.10,wf_2023_2024,20.436335,55.654474,-19.385503,0.889839,10.276470,0.130533,False
2,cap10_release_0.15__wf_2022,cap10_release_0.15,0.15,wf_2022,-1.203459,-76.614257,71.918219,-0.012868,10.014948,0.156447,True
3,cap10_release_0.15__wf_2023_2024,cap10_release_0.15,0.15,wf_2023_2024,19.497958,55.654474,-20.384999,0.865815,10.990701,0.129087,False
4,cap10_release_0.20__wf_2022,cap10_release_0.20,0.20,wf_2022,-1.027383,-76.614257,68.074895,0.015813,12.051659,0.158553,True
5,cap10_release_0.20__wf_2023_2024,cap10_release_0.20,0.20,wf_2023_2024,28.989484,55.654474,-19.944298,1.261319,8.851442,0.128237,False
6,cap10_release_0.25__wf_2022,cap10_release_0.25,0.25,wf_2022,-12.260703,-76.614257,55.207557,-0.584832,19.092674,0.162237,False
7,cap10_release_0.25__wf_2023_2024,cap10_release_0.25,0.25,wf_2023_2024,28.070650,55.654474,-19.269810,1.230196,8.851442,0.129135,False
8,cap10_release_0.30__wf_2022,cap10_release_0.30,0.30,wf_2022,-10.649173,-76.614257,72.410846,-0.904598,12.251923,0.097237,True
9,cap10_release_0.30__wf_2023_2024,cap10_release_0.30,0.30,wf_2023_2024,20.436335,55.654474,-20.821938,0.889839,10.276470,0.128682,False


,cluster_id,cluster_start,cluster_end,profile_count,window_count,profiles,mean_excess_return,worst_excess_return,mean_max_dd,max_max_dd,mean_avg_pos,mean_bad_score,max_bad_score,dominant_failure_type,primary_optimization_path
0,C003,2023-10-23,2024-03-07,6,48,"cap10_release_0.10__wf_2023_2024,cap10_release...",-112.967255,-130.537576,5.151819,5.151819,0.106834,142.967255,160.537576,underparticipation,reentry_participation_or_behavior_clone
1,C002,2022-09-02,2022-12-29,6,33,"cap10_release_0.10__wf_2022,cap10_release_0.15...",-8.070640,-17.893409,6.808061,9.060112,0.018385,38.070640,47.893409,underparticipation,reentry_participation_or_behavior_clone
2,C001,2022-05-09,2022-08-18,5,15,"cap10_release_0.10__wf_2022,cap10_release_0.15...",-4.818807,-10.913696,8.937712,14.739362,0.080653,34.818807,40.913696,underparticipation,reentry_participation_or_behavior_clone


,feature_group,feature_count,mean_abs_delta_z,max_stability_score,mean_spearman_to_excess,top_features
0,regime_or_drawdown,6,0.947317,4.694082,-0.017558,"rolling_new_low_ratio_63,regime_risk,drawdown_..."
1,volatility_or_downside,12,0.870869,4.478630,0.079657,"realized_kurtosis_21,vol_coef_21,regime_risk_n..."
2,trend_or_return,12,0.851141,3.490700,-0.057655,"anchor_slope_180,below_anchor_ratio_63,anchor_..."
3,soxx_relative_or_tracking_error,3,0.802526,2.467018,-0.008355,"tracking_error_vol_21,relative_strength_21,tra..."
4,other,7,0.695407,1.608011,-0.107401,"cum_drop_30,rsi_14,cum_drop_20,days_since_high..."


## 2. Candidate Design

The grid is deliberately small. It separates three ideas:

1. `no_overlay_replay`: replay the existing cap10 paths from their position columns.
2. Low-floor participation probes: test whether a 0.18-0.30 floor can reduce the 2023-2024 underparticipation gap without breaking the 15% drawdown target.
3. SOXX/SOXL relative gates: use `relative_strength_21` and `tracking_error_vol_21` as diagnostics for whether SOXX-relative spread/oscillation should become a future training feature.

The screen is not a hidden promotion search. Even if a row scores best, promotion requires independent walk-forward replay and leakage-safe training labels.

In [3]:
CONFIG_SPECS: list[dict[str, Any]] = [
    {
        "candidate": "no_overlay_replay",
        "description": "replay existing cap10 positions from price path; no participation lift",
        "config": None,
    },
    {
        "candidate": "low_floor_rebound_dd10",
        "description": "loose rebound participation floor with equity drawdown lift-block at 10%",
        "config": ReentryParticipationOverlayConfig(
            enabled=True,
            min_target=0.30,
            max_target=0.30,
            max_base_position=0.18,
            min_mom_21=0.00,
            min_relative_strength_21=-1.00,
            min_vol_of_vol_21=0.05,
            min_tracking_error_vol_21=None,
            max_vix_percentile_252=0.95,
            max_equity_drawdown=0.10,
        ),
    },
    {
        "candidate": "low_floor_rebound_dd08",
        "description": "smaller 0.22 floor with tighter 8% equity drawdown lift-block",
        "config": ReentryParticipationOverlayConfig(
            enabled=True,
            min_target=0.22,
            max_target=0.22,
            max_base_position=0.18,
            min_mom_21=0.00,
            min_relative_strength_21=0.00,
            min_vol_of_vol_21=0.05,
            min_tracking_error_vol_21=None,
            max_vix_percentile_252=0.90,
            max_equity_drawdown=0.08,
        ),
    },
    {
        "candidate": "soxx_te_low_floor_dd10",
        "description": "low floor only when SOXX/SOXL tracking-error volatility is elevated",
        "config": ReentryParticipationOverlayConfig(
            enabled=True,
            min_target=0.30,
            max_target=0.30,
            max_base_position=0.18,
            min_mom_21=0.00,
            min_relative_strength_21=-1.00,
            min_vol_of_vol_21=0.05,
            min_tracking_error_vol_21=0.05,
            max_vix_percentile_252=0.95,
            max_equity_drawdown=0.10,
        ),
    },
    {
        "candidate": "soxx_strength_floor30_dd10",
        "description": "low floor gated by positive SOXX-relative strength",
        "config": ReentryParticipationOverlayConfig(
            enabled=True,
            min_target=0.30,
            max_target=0.30,
            max_base_position=0.18,
            min_mom_21=0.02,
            min_relative_strength_21=0.50,
            min_vol_of_vol_21=0.05,
            min_tracking_error_vol_21=None,
            max_vix_percentile_252=0.95,
            max_equity_drawdown=0.10,
        ),
    },
    {
        "candidate": "trend_floor35_dd08",
        "description": "moderate trend participation floor with tight drawdown lift-block",
        "config": ReentryParticipationOverlayConfig(
            enabled=True,
            min_target=0.35,
            max_target=0.45,
            max_base_position=0.20,
            min_mom_21=0.02,
            min_relative_strength_21=0.00,
            min_vol_of_vol_21=0.05,
            min_tracking_error_vol_21=None,
            max_vix_percentile_252=0.85,
            max_equity_drawdown=0.08,
        ),
    },
    {
        "candidate": "trend_floor45_dd08",
        "description": "higher trend floor; expected to improve upside but risk drawdown breach",
        "config": ReentryParticipationOverlayConfig(
            enabled=True,
            min_target=0.45,
            max_target=0.55,
            max_base_position=0.25,
            min_mom_21=0.05,
            min_relative_strength_21=0.50,
            min_vol_of_vol_21=0.06,
            min_tracking_error_vol_21=None,
            max_vix_percentile_252=0.85,
            max_equity_drawdown=0.08,
        ),
    },
    {
        "candidate": "strict_soxx_te45_dd15",
        "description": "strict SOXX tracking-error and relative-strength confirmation",
        "config": ReentryParticipationOverlayConfig(
            enabled=True,
            min_target=0.45,
            max_target=0.55,
            max_base_position=0.25,
            min_mom_21=0.12,
            min_relative_strength_21=1.00,
            min_vol_of_vol_21=0.10,
            min_tracking_error_vol_21=0.06,
            max_vix_percentile_252=0.85,
            max_equity_drawdown=0.15,
        ),
    },
    {
        "candidate": "permissive_floor35_dd12",
        "description": "permissive floor to test upside ceiling and drawdown cost",
        "config": ReentryParticipationOverlayConfig(
            enabled=True,
            min_target=0.35,
            max_target=0.45,
            max_base_position=0.35,
            min_mom_21=0.00,
            min_relative_strength_21=0.00,
            min_vol_of_vol_21=0.05,
            min_tracking_error_vol_21=None,
            max_vix_percentile_252=0.95,
            max_equity_drawdown=0.12,
        ),
    },
]

config_rows = []
for spec in CONFIG_SPECS:
    cfg = spec["config"]
    row = {"candidate": spec["candidate"], "description": spec["description"]}
    if cfg is None:
        row.update({"enabled": False})
    else:
        row.update(asdict(cfg))
    config_rows.append(row)
config_df = pd.DataFrame(config_rows)
config_df.to_csv(CARD / "candidate_configs.csv", index=False)
display(config_df)


,candidate,description,enabled,min_target,max_target,max_base_position,min_mom_21,min_relative_strength_21,min_vol_of_vol_21,min_tracking_error_vol_21,max_vix_percentile_252,max_equity_drawdown
0,no_overlay_replay,replay existing cap10 positions from price pat...,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,low_floor_rebound_dd10,loose rebound participation floor with equity ...,True,0.30,0.30,0.18,0.00,-1.0,0.05,NaN,0.95,0.10
2,low_floor_rebound_dd08,smaller 0.22 floor with tighter 8% equity draw...,True,0.22,0.22,0.18,0.00,0.0,0.05,NaN,0.90,0.08
3,soxx_te_low_floor_dd10,low floor only when SOXX/SOXL tracking-error v...,True,0.30,0.30,0.18,0.00,-1.0,0.05,0.05,0.95,0.10
4,soxx_strength_floor30_dd10,low floor gated by positive SOXX-relative stre...,True,0.30,0.30,0.18,0.02,0.5,0.05,NaN,0.95,0.10
5,trend_floor35_dd08,moderate trend participation floor with tight ...,True,0.35,0.45,0.20,0.02,0.0,0.05,NaN,0.85,0.08
6,trend_floor45_dd08,higher trend floor; expected to improve upside...,True,0.45,0.55,0.25,0.05,0.5,0.06,NaN,0.85,0.08
7,strict_soxx_te45_dd15,strict SOXX tracking-error and relative-streng...,True,0.45,0.55,0.25,0.12,1.0,0.10,0.06,0.85,0.15
8,permissive_floor35_dd12,permissive floor to test upside ceiling and dr...,True,0.35,0.45,0.35,0.00,0.0,0.05,NaN,0.95,0.12


## 3. Replay And Score

All candidates are replayed from the same daily `position` and `price` columns via `dashboard_daily_frame()`. This makes the path-level comparison apples-to-apples, but it also means the result is a diagnostic replay rather than broker/account truth.

In [4]:
def replay_daily_path(daily: pd.DataFrame, config: ReentryParticipationOverlayConfig | None) -> pd.DataFrame:
    base = daily.copy()
    base["datetime"] = pd.to_datetime(base["datetime"])
    price = base.set_index("datetime")["price"].astype(float)
    if config is None:
        out = dashboard_daily_frame(base[["datetime", "position"]], price)
        out["base_position"] = base["position"].to_numpy(dtype=float)
        out["reentry_participation_feature_ready"] = False
        out["reentry_participation_equity_allowed"] = True
        out["reentry_participation_ready"] = False
        out["reentry_participation_lifted"] = False
        out["equity_nav"] = out["strategy_nav"] / float(out["strategy_nav"].iloc[0])
        out["equity_drawdown"] = 0.0
        return out

    overlay = apply_reentry_participation_overlay(base, features, config)
    out = dashboard_daily_frame(overlay[["datetime", "position"]], overlay.set_index("datetime")["price"].astype(float))
    audit_cols = [
        "datetime",
        "base_position",
        "reentry_participation_feature_ready",
        "reentry_participation_equity_allowed",
        "reentry_participation_ready",
        "reentry_participation_lifted",
        "equity_nav",
        "equity_drawdown",
        "mom_21",
        "relative_strength_21",
        "vol_of_vol_21",
        "tracking_error_vol_21",
        "vix_percentile_252",
    ]
    audit = overlay[[col for col in audit_cols if col in overlay.columns]].copy()
    return out.merge(audit, on="datetime", how="left")


def aggregate_cluster_metrics(rows: list[dict[str, Any]]) -> dict[str, Any]:
    if not rows:
        return {
            "rows": 0,
            "start": "",
            "end": "",
            "return": 0.0,
            "buy_and_hold_return": 0.0,
            "relative_return": float("-inf"),
            "max_dd": float("inf"),
            "avg_pos": 0.0,
        }
    frame = pd.DataFrame(rows)
    return {
        "rows": int(frame["rows"].sum()),
        "start": str(frame["start"].min()),
        "end": str(frame["end"].max()),
        "return": float(frame["return"].mean()),
        "buy_and_hold_return": float(frame["buy_and_hold_return"].mean()),
        "relative_return": float(frame["relative_return"].mean()),
        "max_dd": float(frame["max_dd"].max()),
        "avg_pos": float(frame["avg_pos"].mean()),
    }


def parse_profiles(value: str) -> list[str]:
    return [item.strip() for item in str(value).split(",") if item.strip()]


candidate_frames: dict[str, dict[str, pd.DataFrame]] = {}
window_metrics_by_candidate: dict[str, dict[str, dict[str, Any]]] = {}
cluster_metrics_by_candidate: dict[str, dict[str, dict[str, Any]]] = {}
window_rows: list[dict[str, Any]] = []
cluster_detail_rows: list[dict[str, Any]] = []
cluster_rows: list[dict[str, Any]] = []

for spec in CONFIG_SPECS:
    candidate = spec["candidate"]
    cfg = spec["config"]
    candidate_frames[candidate] = {}
    window_metrics_by_candidate[candidate] = {}

    for manifest_row in manifest.to_dict("records"):
        label = str(manifest_row["label"])
        daily = pd.read_csv(ROOT / str(manifest_row["daily_csv"]), parse_dates=["datetime"])
        replayed = replay_daily_path(daily, cfg)
        replayed["candidate"] = candidate
        replayed["source_label"] = label
        replayed["fold"] = str(manifest_row["fold"])
        replayed["release_cap"] = float(manifest_row["release_cap"])
        candidate_frames[candidate][label] = replayed

        metrics = daily_path_metrics(replayed)
        lifted_days = int(pd.to_numeric(replayed["reentry_participation_lifted"], errors="coerce").fillna(0).sum())
        ready_days = int(pd.to_numeric(replayed["reentry_participation_ready"], errors="coerce").fillna(0).sum())
        feature_ready_days = int(pd.to_numeric(replayed["reentry_participation_feature_ready"], errors="coerce").fillna(0).sum())
        window_metrics_by_candidate[candidate][label] = metrics
        window_rows.append(
            {
                "candidate": candidate,
                "label": label,
                "target_variant": manifest_row["target_variant"],
                "release_cap": float(manifest_row["release_cap"]),
                "fold": manifest_row["fold"],
                "lifted_days": lifted_days,
                "ready_days": ready_days,
                "feature_ready_days": feature_ready_days,
                **metrics,
            }
        )

    cluster_metrics_by_candidate[candidate] = {}
    for cluster in clusters.to_dict("records"):
        cluster_id = str(cluster["cluster_id"])
        per_label_metrics: list[dict[str, Any]] = []
        for label in parse_profiles(cluster["profiles"]):
            if label not in candidate_frames[candidate]:
                continue
            cmetrics = slice_daily_path_metrics(
                candidate_frames[candidate][label],
                start=str(cluster["cluster_start"]),
                end=str(cluster["cluster_end"]),
            )
            per_label_metrics.append(cmetrics)
            cluster_detail_rows.append(
                {
                    "candidate": candidate,
                    "cluster_id": cluster_id,
                    "label": label,
                    "dominant_failure_type": cluster["dominant_failure_type"],
                    **cmetrics,
                }
            )
        aggregate = aggregate_cluster_metrics(per_label_metrics)
        cluster_metrics_by_candidate[candidate][cluster_id] = aggregate
        cluster_rows.append(
            {
                "candidate": candidate,
                "cluster_id": cluster_id,
                "dominant_failure_type": cluster["dominant_failure_type"],
                "source_profile_count": int(cluster["profile_count"]),
                **aggregate,
            }
        )

window_df = pd.DataFrame(window_rows)
cluster_detail_df = pd.DataFrame(cluster_detail_rows)
cluster_df = pd.DataFrame(cluster_rows)
window_df.to_csv(CARD / "window_metrics.csv", index=False)
cluster_df.to_csv(CARD / "cluster_metrics.csv", index=False)
cluster_detail_df.to_csv(CARD / "cluster_detail_metrics.csv", index=False)

display(window_df.groupby(["candidate", "fold"]).agg(
    relative_min=("relative_return", "min"),
    relative_mean=("relative_return", "mean"),
    max_dd_max=("max_dd", "max"),
    avg_pos_mean=("avg_pos", "mean"),
    lifted_days=("lifted_days", "sum"),
).reset_index().round(4))
display(cluster_df.round(4))


,candidate,fold,relative_min,relative_mean,max_dd_max,avg_pos_mean,lifted_days
0,low_floor_rebound_dd08,wf_2022,67.0579,71.1096,21.1580,0.0788,95
1,low_floor_rebound_dd08,wf_2023_2024,-166.0673,-164.2092,14.6664,0.1474,1137
2,low_floor_rebound_dd10,wf_2022,65.2690,69.5965,22.0224,0.0924,153
3,low_floor_rebound_dd10,wf_2023_2024,-143.5791,-141.8895,15.1243,0.1793,1161
4,no_overlay_replay,wf_2022,68.6435,75.3344,18.6781,0.0687,0
5,no_overlay_replay,wf_2023_2024,-193.1040,-190.5536,14.6664,0.0958,0
6,permissive_floor35_dd12,wf_2022,61.9734,65.2270,25.4053,0.0894,114
7,permissive_floor35_dd12,wf_2023_2024,-135.2585,-131.6112,19.6615,0.2022,1257
8,soxx_strength_floor30_dd10,wf_2022,63.8941,67.5982,23.3850,0.0808,77
9,soxx_strength_floor30_dd10,wf_2023_2024,-162.7499,-161.0738,16.8827,0.1705,1032


,candidate,cluster_id,dominant_failure_type,source_profile_count,rows,start,end,return,buy_and_hold_return,relative_return,max_dd,avg_pos
0,no_overlay_replay,C003,underparticipation,6,564,2023-10-23,2024-03-07,17.8864,237.5229,-219.6365,2.4806,0.1006
1,no_overlay_replay,C002,underparticipation,6,492,2022-09-02,2022-12-29,-2.2219,-24.8056,22.5837,7.5036,0.0375
2,no_overlay_replay,C001,underparticipation,5,355,2022-05-09,2022-08-18,5.8727,5.9331,-0.0604,7.4252,0.0936
3,low_floor_rebound_dd10,C003,underparticipation,6,564,2023-10-23,2024-03-07,43.4423,237.5229,-194.0806,4.3724,0.2513
4,low_floor_rebound_dd10,C002,underparticipation,6,492,2022-09-02,2022-12-29,-2.2219,-24.8056,22.5837,7.5036,0.0375
5,low_floor_rebound_dd10,C001,underparticipation,5,355,2022-05-09,2022-08-18,2.9416,5.9331,-2.9915,11.6852,0.1476
6,low_floor_rebound_dd08,C003,underparticipation,6,564,2023-10-23,2024-03-07,29.9189,237.5229,-207.6040,3.2153,0.1895
7,low_floor_rebound_dd08,C002,underparticipation,6,492,2022-09-02,2022-12-29,-2.2219,-24.8056,22.5837,7.5036,0.0375
8,low_floor_rebound_dd08,C001,underparticipation,5,355,2022-05-09,2022-08-18,4.1049,5.9331,-1.8281,7.4252,0.1133
9,soxx_te_low_floor_dd10,C003,underparticipation,6,564,2023-10-23,2024-03-07,24.7201,237.5229,-212.8028,2.9452,0.1403


In [5]:
objective_config = StateConditionedObjectiveConfig(
    target_relative_return=TARGET_RELATIVE_RETURN,
    target_max_dd=TARGET_MAX_DD,
    cluster_relative_floor=-30.0,
    cluster_participation_floor=CLUSTER_PARTICIPATION_FLOOR,
    cluster_max_dd_gap_weight=5.0,
)

scores = score_state_conditioned_objective(
    window_metrics_by_candidate,
    cluster_metrics_by_candidate=cluster_metrics_by_candidate,
    config=objective_config,
)
summary = summarize_state_conditioned_objective(scores)
scores.to_csv(CARD / "objective_scores.csv", index=False)
review = render_state_conditioned_objective_review(scores, summary)
(CARD / "objective_review.md").write_text(review, encoding="utf-8")

display(scores)
display(Markdown(review))


,candidate,window_count,window_pass_count,window_gate,cluster_count,cluster_gate,objective_pass,objective_loss,relative_loss,drawdown_loss,participation_loss,min_relative_return,max_max_dd,mean_avg_pos,worst_window,worst_cluster
0,low_floor_rebound_dd10,12,0,False,3,False,False,1137.826042,937.583511,168.759616,31.482915,-143.579076,22.022415,0.135877,cap10_release_0.25__wf_2023_2024,C003
1,low_floor_rebound_dd08,12,1,False,3,False,False,1200.406474,1051.545357,107.900185,40.960932,-166.067266,21.157998,0.113105,cap10_release_0.25__wf_2023_2024,C003
2,soxx_te_low_floor_dd10,12,0,False,3,False,False,1261.265983,1079.742993,139.283292,42.239697,-170.624900,21.073647,0.116506,cap10_release_0.25__wf_2023_2024,C003
3,no_overlay_replay,12,4,False,3,False,False,1282.887653,1182.127484,48.931259,51.828910,-193.103999,18.678084,0.082236,cap10_release_0.15__wf_2023_2024,C003
4,soxx_strength_floor30_dd10,12,0,False,3,False,False,1352.209117,1030.383715,286.375276,35.450126,-162.749881,23.385024,0.125655,cap10_release_0.10__wf_2023_2024,C003
5,permissive_floor35_dd12,12,0,False,3,False,False,1357.120455,884.127742,440.169433,32.823280,-135.258465,25.405292,0.145821,cap10_release_0.10__wf_2023_2024,C003
6,trend_floor35_dd08,12,1,False,3,False,False,1401.108836,1064.370438,301.388531,35.349867,-170.649171,23.114103,0.122230,cap10_release_0.15__wf_2023_2024,C003
7,strict_soxx_te45_dd15,12,2,False,3,False,False,1463.039688,1196.961422,219.159324,46.918942,-195.803149,21.875399,0.095814,cap10_release_0.15__wf_2023_2024,C003
8,trend_floor45_dd08,12,1,False,3,False,False,1562.530040,1096.128430,418.985175,47.416435,-175.076852,22.812781,0.105903,cap10_release_0.15__wf_2023_2024,C003


# State-Conditioned Objective Screen

## Decision

- decision: `no_objective_pass_candidate`
- selected_candidate: `low_floor_rebound_dd10`
- next_mechanism: `train_state_conditioned_multi_objective`

## Ranking

| candidate | pass | loss | wf pass | cluster pass | rel loss | dd loss | part loss | worst wf | worst cluster |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | --- | --- |
| low_floor_rebound_dd10 | False | 1137.83 | 0/12 | False | 937.58 | 168.76 | 31.48 | cap10_release_0.25__wf_2023_2024 | C003 |
| low_floor_rebound_dd08 | False | 1200.41 | 1/12 | False | 1051.55 | 107.90 | 40.96 | cap10_release_0.25__wf_2023_2024 | C003 |
| soxx_te_low_floor_dd10 | False | 1261.27 | 0/12 | False | 1079.74 | 139.28 | 42.24 | cap10_release_0.25__wf_2023_2024 | C003 |
| no_overlay_replay | False | 1282.89 | 4/12 | False | 1182.13 | 48.93 | 51.83 | cap10_release_0.15__wf_2023_2024 | C003 |
| soxx_strength_floor30_dd10 | False | 1352.21 | 0/12 | False | 1030.38 | 286.38 | 35.45 | cap10_release_0.10__wf_2023_2024 | C003 |
| permissive_floor35_dd12 | False | 1357.12 | 0/12 | False | 884.13 | 440.17 | 32.82 | cap10_release_0.10__wf_2023_2024 | C003 |
| trend_floor35_dd08 | False | 1401.11 | 1/12 | False | 1064.37 | 301.39 | 35.35 | cap10_release_0.15__wf_2023_2024 | C003 |
| strict_soxx_te45_dd15 | False | 1463.04 | 2/12 | False | 1196.96 | 219.16 | 46.92 | cap10_release_0.15__wf_2023_2024 | C003 |
| trend_floor45_dd08 | False | 1562.53 | 1/12 | False | 1096.13 | 418.99 | 47.42 | cap10_release_0.15__wf_2023_2024 | C003 |

## Readout

- A passing row is a promotion-review input, not an automatic runtime bind.
- If no row passes, treat the selected row as a diagnostic target and keep no runtime promotion.
- Required terms should be implemented in training-time reward/checkpoint selection before another path-level threshold search.


## 4. Persist Selection

The selected candidate is the lowest objective-loss diagnostic row, not a promotion. This cell writes the machine-readable sidecars and exports only the selected candidate daily paths for audit.

In [6]:
def jsonable(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): jsonable(v) for k, v in value.items()}
    if isinstance(value, list):
        return [jsonable(v) for v in value]
    if isinstance(value, tuple):
        return [jsonable(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        if np.isnan(value):
            return None
        if np.isposinf(value):
            return "inf"
        if np.isneginf(value):
            return "-inf"
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, float):
        if np.isnan(value):
            return None
        if np.isposinf(value):
            return "inf"
        if np.isneginf(value):
            return "-inf"
        return value
    return value

selected = scores.iloc[0].to_dict() if not scores.empty else {}
selected_candidate = str(selected.get("candidate", ""))
baseline_candidate = "no_overlay_replay"

selected_paths = []
for label, frame in candidate_frames.get(selected_candidate, {}).items():
    out_path = SELECTED_DAILY_DIR / f"{selected_candidate}__{label}.csv"
    frame.to_csv(out_path, index=False)
    selected_paths.append({"candidate": selected_candidate, "label": label, "path": str(out_path.relative_to(ROOT)), "rows": int(len(frame))})
selected_path_manifest = pd.DataFrame(selected_paths)
selected_path_manifest.to_csv(CARD / "selected_path_manifest.csv", index=False)

selected_window_metrics = window_df[window_df["candidate"] == selected_candidate].copy()
baseline_window_metrics = window_df[window_df["candidate"] == baseline_candidate].copy()
comparison = selected_window_metrics.merge(
    baseline_window_metrics,
    on=["label", "target_variant", "release_cap", "fold"],
    suffixes=("_selected", "_baseline"),
)
for col in ["return", "relative_return", "max_dd", "avg_pos", "lifted_days", "ready_days"]:
    comparison[f"delta_{col}"] = comparison[f"{col}_selected"] - comparison[f"{col}_baseline"]
comparison.to_csv(CARD / "selected_vs_no_overlay.csv", index=False)

selected_config = next(spec for spec in CONFIG_SPECS if spec["candidate"] == selected_candidate)
if bool(selected.get("objective_pass", False)):
    decision = "objective_pass_independent_promotion_review_required"
    blocker = "Path-level objective passed, but independent walk-forward/promotion review is still required."
else:
    decision = "diagnostic_no_objective_pass_no_promotion"
    blocker = "No candidate jointly cleared +30pp relative return, <=15% MaxDD, and cluster participation/drawdown gates."

selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "source_experiment": "cap10_release_oos_poor_window_diagnostic_v1",
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": "sparse_release_cap10_rebound_balance_v1",
    "role": "diagnose",
    "changed_layer": ["model_selection", "training_objective", "path_replay"],
    "decision": decision,
    "promotion_allowed": False,
    "full_gate": bool(selected.get("objective_pass", False)),
    "promotion_blocker": blocker,
    "selected_candidate": selected_candidate,
    "selected_config": None if selected_config["config"] is None else asdict(selected_config["config"]),
    "selected_description": selected_config["description"],
    "selected_row": selected,
    "objective_summary": summary,
    "selected_window_metrics": selected_window_metrics.to_dict("records"),
    "selected_cluster_metrics": cluster_df[cluster_df["candidate"] == selected_candidate].to_dict("records"),
    "selected_vs_no_overlay": comparison.to_dict("records"),
    "external_sources": [
        "https://github.com/microsoft/qlib",
        "https://arxiv.org/abs/2009.11189",
        "https://doi.org/10.21314/JCF.2016.322",
        "https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551",
        "https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2708678",
    ],
    "sidecars": [
        "candidate_configs.csv",
        "window_metrics.csv",
        "cluster_metrics.csv",
        "cluster_detail_metrics.csv",
        "objective_scores.csv",
        "objective_review.md",
        "selected_vs_no_overlay.csv",
        "selected_path_manifest.csv",
        "summary.json",
    ],
    "acceptance_gate_for_next_run": {
        "relative_return_min_pp": TARGET_RELATIVE_RETURN,
        "max_dd_max_pct": TARGET_MAX_DD,
        "cluster_participation_floor": CLUSTER_PARTICIPATION_FLOOR,
        "fixed_split_required": True,
        "walkforward_required": True,
        "purged_or_fold_local_labels_required_before_training": True,
        "no_runtime_promotion_from_path_level_screen": True,
    },
}

summary_doc = {
    "experiment": EXPERIMENT,
    "decision": decision,
    "selected_candidate": selected_candidate,
    "selected_objective_loss": selected.get("objective_loss"),
    "selected_window_gate": selected.get("window_gate"),
    "selected_cluster_gate": selected.get("cluster_gate"),
    "selected_worst_window": selected.get("worst_window"),
    "selected_worst_cluster": selected.get("worst_cluster"),
    "selected_min_relative_return": selected.get("min_relative_return"),
    "selected_max_max_dd": selected.get("max_max_dd"),
    "selected_mean_avg_pos": selected.get("mean_avg_pos"),
    "conclusion": [
        "A low participation floor can improve the 2023-2024 underparticipation gap versus the no-overlay replay, but it still leaves relative return far below +30pp and breaches or nearly consumes the 15% drawdown budget.",
        "SOXX/SOXL tracking-error and relative-strength gates did not produce a full-gate candidate in this path-level screen.",
        "Treat the result as a training-objective clue: future labels should be fold-local, drawdown-aware, and evaluated by walk-forward checkpoint selection before any PPO or runtime promotion.",
    ],
}

(CARD / "selection.json").write_text(json.dumps(jsonable(selection), ensure_ascii=True, indent=2), encoding="utf-8")
(CARD / "summary.json").write_text(json.dumps(jsonable(summary_doc), ensure_ascii=True, indent=2), encoding="utf-8")

assert len(window_df) == len(CONFIG_SPECS) * len(manifest)
assert len(scores) == len(CONFIG_SPECS)
assert (CARD / "selection.json").exists()
assert selection["promotion_allowed"] is False

print(json.dumps(jsonable(summary_doc), indent=2))


{
  "experiment": "cap10_state_conditioned_trend_participation_controller_v1",
  "decision": "diagnostic_no_objective_pass_no_promotion",
  "selected_candidate": "low_floor_rebound_dd10",
  "selected_objective_loss": 1137.8260420315696,
  "selected_window_gate": false,
  "selected_cluster_gate": false,
  "selected_worst_window": "cap10_release_0.25__wf_2023_2024",
  "selected_worst_cluster": "C003",
  "selected_min_relative_return": -143.5790761444367,
  "selected_max_max_dd": 22.02241462860518,
  "selected_mean_avg_pos": 0.1358771157192579,
  "conclusion": [
    "A low participation floor can improve the 2023-2024 underparticipation gap versus the no-overlay replay, but it still leaves relative return far below +30pp and breaches or nearly consumes the 15% drawdown budget.",
    "SOXX/SOXL tracking-error and relative-strength gates did not produce a full-gate candidate in this path-level screen.",
    "Treat the result as a training-objective clue: future labels should be fold-local, 

## 5. Runtime Conclusion

This cell is intentionally generated after execution from the persisted selection so the notebook conclusion and sidecar stay aligned.

In [7]:
selection = json.loads((CARD / "selection.json").read_text())
summary_doc = json.loads((CARD / "summary.json").read_text())
selected_candidate = selection["selected_candidate"]
selected = selection["selected_row"]
comparison = pd.read_csv(CARD / "selected_vs_no_overlay.csv")
fold_summary = comparison.groupby("fold").agg(
    selected_relative_min=("relative_return_selected", "min"),
    selected_relative_mean=("relative_return_selected", "mean"),
    baseline_relative_mean=("relative_return_baseline", "mean"),
    delta_relative_mean=("delta_relative_return", "mean"),
    selected_max_dd_max=("max_dd_selected", "max"),
    baseline_max_dd_max=("max_dd_baseline", "max"),
    selected_avg_pos_mean=("avg_pos_selected", "mean"),
    baseline_avg_pos_mean=("avg_pos_baseline", "mean"),
    lifted_days_sum=("lifted_days_selected", "sum"),
).reset_index()
fold_summary.to_csv(CARD / "selected_fold_summary.csv", index=False)

display(fold_summary.round(4))

conclusion = f"""
### Decision: `{selection['decision']}`

Selected diagnostic row: `{selected_candidate}` with objective loss `{selected.get('objective_loss'):.2f}`. It does **not** pass the full objective gate: window gate `{selected.get('window_gate')}`, cluster gate `{selected.get('cluster_gate')}`. Worst window is `{selected.get('worst_window')}` and worst cluster is `{selected.get('worst_cluster')}`.

Key readout: the selected low-floor participation controller raises average participation versus the no-overlay replay, but the worst relative return remains `{selected.get('min_relative_return'):.2f}pp` and the maximum path drawdown reaches `{selected.get('max_max_dd'):.2f}%`. That misses the target of `+30pp` relative return and `<=15%` MaxDD, so there is no promotion and no runtime overlay handoff.

SOXX/SOXL factor note: candidates using `relative_strength_21` / `tracking_error_vol_21` did not clear the joint gate. Keep the SOXX/SOXL spread idea as a training-state feature hypothesis, not as a direct post-hoc threshold overlay.

Next action: build fold-local training labels or a checkpoint-selection loss that rewards rebound participation only when the sequence-risk state keeps projected drawdown inside the 15% budget. Apply purged/fold-local validation before PPO because this screen was selected from OOS daily paths.

Artifacts: `candidate_configs.csv`, `window_metrics.csv`, `cluster_metrics.csv`, `objective_scores.csv`, `selected_vs_no_overlay.csv`, `selected_fold_summary.csv`, `selected_path_manifest.csv`, `selection.json`, and selected daily paths under `selected_daily_paths/`.
"""
display(Markdown(conclusion))


,fold,selected_relative_min,selected_relative_mean,baseline_relative_mean,delta_relative_mean,selected_max_dd_max,baseline_max_dd_max,selected_avg_pos_mean,baseline_avg_pos_mean,lifted_days_sum
0,wf_2022,65.2690,69.5965,75.3344,-5.7379,22.0224,18.6781,0.0924,0.0687,153
1,wf_2023_2024,-143.5791,-141.8895,-190.5536,48.6640,15.1243,14.6664,0.1793,0.0958,1161



### Decision: `diagnostic_no_objective_pass_no_promotion`

Selected diagnostic row: `low_floor_rebound_dd10` with objective loss `1137.83`. It does **not** pass the full objective gate: window gate `False`, cluster gate `False`. Worst window is `cap10_release_0.25__wf_2023_2024` and worst cluster is `C003`.

Key readout: the selected low-floor participation controller raises average participation versus the no-overlay replay, but the worst relative return remains `-143.58pp` and the maximum path drawdown reaches `22.02%`. That misses the target of `+30pp` relative return and `<=15%` MaxDD, so there is no promotion and no runtime overlay handoff.

SOXX/SOXL factor note: candidates using `relative_strength_21` / `tracking_error_vol_21` did not clear the joint gate. Keep the SOXX/SOXL spread idea as a training-state feature hypothesis, not as a direct post-hoc threshold overlay.

Next action: build fold-local training labels or a checkpoint-selection loss that rewards rebound participation only when the sequence-risk state keeps projected drawdown inside the 15% budget. Apply purged/fold-local validation before PPO because this screen was selected from OOS daily paths.

Artifacts: `candidate_configs.csv`, `window_metrics.csv`, `cluster_metrics.csv`, `objective_scores.csv`, `selected_vs_no_overlay.csv`, `selected_fold_summary.csv`, `selected_path_manifest.csv`, `selection.json`, and selected daily paths under `selected_daily_paths/`.
